# BLIP 基础教程

**BLIP** (Bootstrapped Language-Image Pre-training) 是 Salesforce 提出的多模态模型。

## 你已经学过 CLIP，为什么还要学 BLIP？

回忆一下，CLIP 是一个**双编码器**架构 —— 图像编码器和文本编码器各自独立工作，最后通过余弦相似度匹配。它擅长"理解"（这张图和这段文字有多像？），但**不能生成文字**。

BLIP 的突破在于：**同一个模型，既能理解，又能生成。**

| 能力 | CLIP | BLIP |
|------|------|------|
| 图文匹配 (ITM) | ✅ 核心能力 | ✅ |
| 图像描述生成 | ❌ | ✅ 核心能力 |
| 视觉问答 (VQA) | ❌ | ✅ |
| 架构 | 双编码器 | 编码器 + 解码器 |

本教程从最简单的用法开始，带你一步步体验 BLIP 的三大能力。

## Part 1: 环境准备

和 CLIP 教程一样，我们先导入必要的库。

In [ ]:
import torch
from PIL import Image
import requests
import matplotlib.pyplot as plt

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Part 2: 准备测试图片

我们准备两张图片，后面所有实验都用它们。

In [ ]:
# BLIP 官方 demo 图片：一个女人和一条狗在海滩上
url1 = "https://storage.googleapis.com/sfr-vision-language-research/BLIP/demo.jpg"
# 一只胖猫
url2 = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/pipeline-cat-chonk.jpeg"

image1 = Image.open(requests.get(url1, stream=True).raw).convert("RGB")
image2 = Image.open(requests.get(url2, stream=True).raw).convert("RGB")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(image1)
axes[0].set_title("Image 1")
axes[0].axis("off")
axes[1].imshow(image2)
axes[1].set_title("Image 2")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
---

## Part 3: 图像描述生成 (Image Captioning)

这是 BLIP 最核心的能力，也是 CLIP 做不到的事情 —— **看一张图，自动写出一句描述。**

BLIP 用的是 `BlipForConditionalGeneration`，它内部包含：
- **Vision Encoder**：把图像变成特征向量（和 CLIP 的 Image Encoder 类似）
- **Text Decoder**：根据图像特征，一个词一个词地生成描述

我们用 Salesforce 官方的 `blip-image-captioning-base` 模型。

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration

# 加载 captioning 模型（约 990MB）
caption_model_name = "Salesforce/blip-image-captioning-base"
print(f"正在加载模型: {caption_model_name}")

caption_processor = BlipProcessor.from_pretrained(caption_model_name)
caption_model = BlipForConditionalGeneration.from_pretrained(caption_model_name).to(device)
caption_model.eval()

print("模型加载完成！")
print(f"模型参数量: {sum(p.numel() for p in caption_model.parameters()) / 1e6:.1f}M")

### 3.1 无条件生成 —— 让模型自由描述

最简单的用法：只给图片，不给任何提示，让模型自己决定说什么。

In [ ]:
# 无条件生成：只传图片，不传文本
for i, image in enumerate([image1, image2], 1):
    inputs = caption_processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = caption_model.generate(**inputs, max_length=50)

    caption = caption_processor.decode(output_ids[0], skip_special_tokens=True)
    print(f"Image {i}: {caption}")

### 3.2 条件生成 —— 给模型一个开头

你可以给模型一个文本开头（prompt），让它接着往下写。这很像"填空题"：

> **你给：** "a woman and her dog..."
> **模型续写：** "a woman and her dog are sitting on the beach"

这在实际应用中很有用 —— 比如你想让描述以特定风格开头。

In [ ]:
# 条件生成：给一个文本开头，模型接着写
prompts = [
    "a photography of",
    "this is a scene where",
    "in this image, we can see",
]

print("=== Image 1: 海滩场景 ===")
for prompt in prompts:
    inputs = caption_processor(images=image1, text=prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = caption_model.generate(**inputs, max_length=50)

    caption = caption_processor.decode(output_ids[0], skip_special_tokens=True)
    print(f"  Prompt: \"{prompt}\"")
    print(f"  Output: {caption}\n")

### 3.3 生成策略对比

和 LLM 一样，图像描述生成也有不同的解码策略。我们来对比三种：

| 策略 | 特点 | 适用场景 |
|------|------|---------|
| **Greedy** | 每步选概率最高的词，速度最快 | 快速推理 |
| **Beam Search** | 同时保留多条候选路径，质量更高 | 追求准确性 |
| **Sampling** | 按概率分布随机采样，结果多样 | 需要多样性 |

In [ ]:
inputs = caption_processor(images=image1, return_tensors="pt").to(device)

with torch.no_grad():
    # 1) Greedy：最快，但结果单一
    greedy_ids = caption_model.generate(**inputs, max_length=50, num_beams=1, do_sample=False)

    # 2) Beam Search：质量更高
    beam_ids = caption_model.generate(**inputs, max_length=50, num_beams=5, do_sample=False)

    # 3) Sampling：每次结果可能不同
    sample_ids = caption_model.generate(
        **inputs, max_length=50, do_sample=True,
        temperature=0.7, top_k=50, top_p=0.95,
    )

print("Greedy:      ", caption_processor.decode(greedy_ids[0], skip_special_tokens=True))
print("Beam Search: ", caption_processor.decode(beam_ids[0], skip_special_tokens=True))
print("Sampling:    ", caption_processor.decode(sample_ids[0], skip_special_tokens=True))

---

## Part 4: 视觉问答 (VQA)

BLIP 的第二大能力：你给一张图和一个问题，模型回答。

这里要用另一个专门的类 `BlipForQuestionAnswering`，和 captioning 模型的区别是：
- Captioning 模型的 decoder 是**自回归生成**（一个词接一个词写下去）
- VQA 模型的 decoder 接收**问题作为输入**，生成**答案**

In [ ]:
from transformers import BlipForQuestionAnswering

vqa_model_name = "Salesforce/blip-vqa-base"
print(f"正在加载 VQA 模型: {vqa_model_name}")

vqa_processor = BlipProcessor.from_pretrained(vqa_model_name)
vqa_model = BlipForQuestionAnswering.from_pretrained(vqa_model_name).to(device)
vqa_model.eval()

print("VQA 模型加载完成！")

In [ ]:
# 对 image1（海滩场景）提问
questions = [
    "What is the woman doing?",
    "What animal is in the picture?",
    "Where are they?",
    "What color is the sky?",
    "How many people are in the image?",
]

print("=== 对 Image 1 提问 ===\n")
for q in questions:
    inputs = vqa_processor(images=image1, text=q, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = vqa_model.generate(**inputs, max_length=20)

    answer = vqa_processor.decode(output_ids[0], skip_special_tokens=True)
    print(f"  Q: {q}")
    print(f"  A: {answer}\n")

In [ ]:
# 对 image2（猫）也试试
cat_questions = [
    "What animal is this?",
    "What is the cat doing?",
    "Is the cat fat?",
]

print("=== 对 Image 2 提问 ===\n")
for q in cat_questions:
    inputs = vqa_processor(images=image2, text=q, return_tensors="pt").to(device)

    with torch.no_grad():
        output_ids = vqa_model.generate(**inputs, max_length=20)

    answer = vqa_processor.decode(output_ids[0], skip_special_tokens=True)
    print(f"  Q: {q}")
    print(f"  A: {answer}\n")

---

## Part 5: 图文匹配 (Image-Text Matching)

BLIP 的第三大能力，也是和 CLIP 最像的部分 —— 判断一张图和一段文字是否匹配。

但 BLIP 的 ITM 比 CLIP 更强：
- **CLIP** 只用独立的编码器分别编码图文，再算余弦相似度（浅层交互）
- **BLIP ITM** 让图像和文本特征在 Transformer 中做 cross-attention（深层交互），能捕捉更细粒度的对应关系

我们用 `BlipForImageTextRetrieval` 来体验。

In [ ]:
from transformers import BlipForImageTextRetrieval

itm_model_name = "Salesforce/blip-itm-base-coco"
print(f"正在加载 ITM 模型: {itm_model_name}")

itm_processor = BlipProcessor.from_pretrained(itm_model_name)
itm_model = BlipForImageTextRetrieval.from_pretrained(itm_model_name).to(device)
itm_model.eval()

print("ITM 模型加载完成！")

In [ ]:
# 测试不同文本和 image1（海滩女人+狗）的匹配程度
candidates = [
    "a woman and a dog on the beach",       # 正确描述
    "a woman sitting with her dog",          # 部分正确
    "a cat sleeping on a bed",              # 完全不相关
    "a dog playing in the ocean",            # 部分相关
    "two men playing basketball",            # 完全不相关
]

print("=== Image 1 与不同文本的匹配度 ===\n")
for text in candidates:
    inputs = itm_processor(images=image1, text=text, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = itm_model(**inputs)

    # itm_score: [not_match, match] 的 logits
    itm_scores = outputs.itm_score
    match_prob = torch.softmax(itm_scores, dim=1)[0, 1].item()

    bar = "█" * int(match_prob * 30) + "░" * (30 - int(match_prob * 30))
    print(f"  {match_prob:.1%} {bar}  \"{text}\"")

---

## Part 6: 三大能力汇总 —— 同一张图，三种玩法

让我们用 image2（猫）同时展示 BLIP 的三种能力，感受它的强大。

In [ ]:
print("=" * 50)
print("  BLIP 三大能力演示 —— Image 2 (猫)")
print("=" * 50)

# 1. Captioning: 描述这张图
inputs = caption_processor(images=image2, return_tensors="pt").to(device)
with torch.no_grad():
    cap_ids = caption_model.generate(**inputs, max_length=50, num_beams=5)
caption = caption_processor.decode(cap_ids[0], skip_special_tokens=True)
print(f"\n📝 Captioning: {caption}")

# 2. VQA: 回答关于这张图的问题
question = "What color is the cat?"
inputs = vqa_processor(images=image2, text=question, return_tensors="pt").to(device)
with torch.no_grad():
    ans_ids = vqa_model.generate(**inputs, max_length=20)
answer = vqa_processor.decode(ans_ids[0], skip_special_tokens=True)
print(f"\n❓ VQA: \"{question}\" → \"{answer}\"")

# 3. ITM: 这段文字和图片匹配吗？
text_match = "a fluffy cat sitting"
text_nomatch = "a dog running in the park"
for text in [text_match, text_nomatch]:
    inputs = itm_processor(images=image2, text=text, return_tensors="pt").to(device)
    with torch.no_grad():
        itm_out = itm_model(**inputs)
    prob = torch.softmax(itm_out.itm_score, dim=1)[0, 1].item()
    print(f"\n🔗 ITM: \"{text}\" → match {prob:.1%}")

---

## Part 7: BLIP 架构原理（选读）

如果你好奇 BLIP 内部是怎么工作的，这里做一个简要说明。

### 三个子任务，共享一个 Vision Encoder

```
                    ┌──────────────────┐
                    │  Vision Encoder  │  ← 所有任务共享，基于 ViT
                    │  (ViT-B/16)      │
                    └────────┬─────────┘
                             │ image features
              ┌──────────────┼──────────────┐
              ▼              ▼              ▼
     ┌────────────┐  ┌──────────────┐  ┌──────────────┐
     │ Text       │  │ Image-grounded│  │ Image-grounded│
     │ Encoder    │  │ Text Encoder  │  │ Text Decoder  │
     │ (ITC)      │  │ (ITM)         │  │ (Captioning)  │
     └────────────┘  └──────────────┘  └──────────────┘
     对比学习         图文匹配           描述生成
     (像 CLIP)       (cross-attention)  (自回归生成)
```

**关键创新 —— CapFilt (Captioning and Filtering)：**

BLIP 的训练数据来自网络，质量参差不齐。它用一个巧妙的"自举"策略：
1. 先在有噪声的数据上训练
2. 用训练好的 captioner 为图片生成新描述
3. 用训练好的 filter (ITM) 过滤掉低质量的图文对
4. 用清洗后的数据重新训练 → 模型变得更好

这就是名字里 "Bootstrapped" 的含义。

---

## Part 8: 总结

本教程我们体验了 BLIP 的三大核心能力：

| 能力 | 模型类 | 用途 |
|------|--------|------|
| **Image Captioning** | `BlipForConditionalGeneration` | 看图写描述 |
| **VQA** | `BlipForQuestionAnswering` | 看图回答问题 |
| **ITM** | `BlipForImageTextRetrieval` | 判断图文是否匹配 |

**和 CLIP 的关系：**
- CLIP 是"理解型"模型（图文匹配、检索），BLIP 在此基础上增加了"生成"能力
- BLIP 的 ITC 部分本质上就是 CLIP 的思路
- BLIP 的 ITM 比 CLIP 更精细（因为有 cross-attention）
- BLIP 独有的 captioning 能力是通向更强大 VLM（如 BLIP-2、LLaVA）的基础

**下一步学习建议：**
- 尝试 `Salesforce/blip-image-captioning-large`，对比 base 和 large 的效果差异
- 尝试用自己的图片测试
- 了解 BLIP-2：它在 BLIP 基础上引入了 Q-Former，进一步连接了冻结的 LLM

## 练习区

试试用你自己的图片和问题来测试！

In [ ]:
# 在这里试试你自己的图片！
# 例如：
# my_image = Image.open("your_image.jpg").convert("RGB")
# inputs = caption_processor(images=my_image, return_tensors="pt").to(device)
# with torch.no_grad():
#     ids = caption_model.generate(**inputs, max_length=50, num_beams=5)
# print(caption_processor.decode(ids[0], skip_special_tokens=True))